# FlyWire Error Analysis: Missed Synapses Experiment

**One Kaggle Run = One Dataset + One Error Model + One Analysis Profile**

---
### How to run on Kaggle
1. Attach both datasets to this notebook:
   - `flywire-codebase` (uploaded from `flywire_codebase.zip`)
   - `flywire-all-datasets` (uploaded from `flywire_all_datasets.zip`)
2. In **Cell 3**, set `DATASET_NAME` to whichever connectome you want to run.
3. Click **Run All**.

### Kaggle Dataset Paths (fixed, no changes needed)
- Codebase : `/kaggle/input/datasets/jeet7771/flywire-codebase`
- Data      : `/kaggle/input/datasets/jeet7771/flywire-all-datasets`

In [ ]:
# Cell 1: Environment Setup & sys.path
# This MUST run before any framework imports.
# ============================================================
import os
import sys
from pathlib import Path

IS_KAGGLE = os.path.exists('/kaggle/input')

# Exact Kaggle dataset mount paths for user jeet7771
KAGGLE_CODEBASE_PATH = Path('/kaggle/input/datasets/jeet7771/flywire-codebase')
KAGGLE_DATA_PATH     = Path('/kaggle/input/datasets/jeet7771/flywire-all-datasets')

if IS_KAGGLE:
    # --- Verify and add codebase to sys.path ---
    if not KAGGLE_CODEBASE_PATH.exists():
        raise FileNotFoundError(
            f'Codebase dataset not found at {KAGGLE_CODEBASE_PATH}\n'
            'Attach the "flywire-codebase" dataset to this notebook.'
        )
    sys.path.insert(0, str(KAGGLE_CODEBASE_PATH))
    print(f'[OK] Codebase path  : {KAGGLE_CODEBASE_PATH}')

    # --- Verify data dataset ---
    if not KAGGLE_DATA_PATH.exists():
        raise FileNotFoundError(
            f'Data dataset not found at {KAGGLE_DATA_PATH}\n'
            'Attach the "flywire-all-datasets" dataset to this notebook.'
        )
    print(f'[OK] Data path      : {KAGGLE_DATA_PATH}')
    print(f'[OK] Datasets found : {[d.name for d in KAGGLE_DATA_PATH.iterdir() if d.is_dir()]}')

else:
    # Local: codebase is the current working directory
    REPO_ROOT = Path(os.getcwd())
    if str(REPO_ROOT) not in sys.path:
        sys.path.insert(0, str(REPO_ROOT))
    print(f'[OK] Running locally. Repo root: {REPO_ROOT}')

print(f'Environment: {"KAGGLE" if IS_KAGGLE else "LOCAL"}')

In [ ]:
# Cell 2: Framework Imports
import warnings
import pandas as pd

warnings.filterwarnings('ignore')

from core.experiment_runner import ExperimentRunner, ExperimentConfig
from modules.error_models.error_registry import registry as error_registry
from modules.graph_analyses.analysis_registry import registry as analysis_registry
from modules.statistical_evaluation import StatisticalEvaluator
from core.export_manager import ExportManager

print('All framework imports successful.')

In [ ]:
# ============================================================
# Cell 3: RUNTIME CONFIGURATION  <-- ONLY CELL YOU NEED TO EDIT
# ============================================================

# Which connectome to run.
# Options: "BANC" | "FAFB" | "MANC" | "MAOL" | "MCNS" | "TEST"
DATASET_NAME = "MANC"

# [LOCAL ONLY] Path to your raw dataset folder. Ignored on Kaggle.
LOCAL_DATASET_ROOT = "research_data/raw"

EXPERIMENT = {
    "metadata": {
        "experiment_name": f"MissedSynapses_{DATASET_NAME}",
        "author": "FlyWire Researcher",
        "description": "Topological degradation from simulated missing synapses.",
    },
    "error": {
        "name": "missed_synapses",
        "rates": [0.00, 0.01, 0.05, 0.10, 0.20],
        "random_seeds": [1, 2, 3, 4, 5],
    },
    "biology": {
        "weights": {
            "synapse_weight": 1.0,
            "source_degree_weight": 0.5,
            "target_degree_weight": 0.5,
        }
    },
    "analysis": [
        "basic_structure",
        "degree_distribution",
        "pagerank",
        # "centrality",  # Temporarily disabled for runtime profiling (Betweenness + Closeness are intractable on large graphs)
        "connected_components",
        "reciprocity",
    ],
    "export": {
        "create_zip": True,
        "save_statistics": True,
    },
}

OUTPUT_ROOT = Path('results') / DATASET_NAME / EXPERIMENT['error']['name']

print(f'Dataset Name    : {DATASET_NAME}')
print(f'Error Model     : {EXPERIMENT["error"]["name"]}')
print(f'Error Rates     : {EXPERIMENT["error"]["rates"]}')
print(f'Trials per Rate : {len(EXPERIMENT["error"]["random_seeds"])}')
print(f'Output Root     : {OUTPUT_ROOT}')

In [ ]:
# Cell 4: Resolve Dataset Root
if IS_KAGGLE:
    DATASET_ROOT = str(KAGGLE_DATA_PATH)
else:
    DATASET_ROOT = '0-demodata' if DATASET_NAME.upper() == 'TEST' else LOCAL_DATASET_ROOT

print(f'DATASET_ROOT = {DATASET_ROOT}')

In [ ]:
# Cell 5: Verify Dataset Structure
from core.dataset_registry import DatasetRegistry, DatasetRegistryError

CONFIGS_ROOT = str(KAGGLE_CODEBASE_PATH / 'configs') if IS_KAGGLE else 'configs'

try:
    reg = DatasetRegistry(configs_root=CONFIGS_ROOT, dataset_root=DATASET_ROOT)
    resolved_dir = reg.resolve_dataset_dir(DATASET_NAME, DATASET_ROOT)
    print(f'[OK] Dataset "{DATASET_NAME}" verified.')
    print(f'     Resolved: {resolved_dir}')
except DatasetRegistryError as e:
    raise FileNotFoundError(
        f'Cannot resolve dataset "{DATASET_NAME}" in "{DATASET_ROOT}".\n'
        f'Expected a subfolder named {DATASET_NAME}_<version>/ or {DATASET_NAME}/.\n'
        f'Error: {e}'
    ) from e

In [ ]:
# Cell 6: Verify Registries
err_model = EXPERIMENT['error']['name']
print(f'Registered Error Models : {error_registry.list_names()}')
print(f'Registered Analyses     : {analysis_registry.list_names()}')

assert err_model in error_registry.list_names(), \
    f'Error model "{err_model}" not registered.'
missing = [a for a in EXPERIMENT['analysis'] if a not in analysis_registry.list_names()]
assert not missing, f'Analyses not registered: {missing}'

print('[OK] All required components registered. Ready to run.')

In [ ]:
# Cell 7: Run Experiments
runner = ExperimentRunner(analysis_registry, error_registry)
results_per_rate = {}

for err_rate in EXPERIMENT['error']['rates']:
    rate_str = f"{int(err_rate * 100)}_percent"
    results_per_rate[err_rate] = []

    for trial, seed in enumerate(EXPERIMENT['error']['random_seeds'], 1):
        print(f'\n{"="*50}')
        print(f'  Dataset    : {DATASET_NAME}')
        print(f'  Error Rate : {err_rate * 100:.1f}%')
        print(f'  Trial      : {trial} / {len(EXPERIMENT["error"]["random_seeds"])}')
        print(f'  Seed       : {seed}')
        print(f'{"="*50}')

        trial_out = OUTPUT_ROOT / rate_str / f'trial_{trial:03d}'

        config = ExperimentConfig(
            dataset_name=DATASET_NAME,
            dataset_root=str(DATASET_ROOT),
            configs_root=CONFIGS_ROOT,
            error_model_name=err_model,
            error_model_config={
                'error_rate': err_rate,
                'biology': EXPERIMENT['biology'],
            },
            analysis_names=EXPERIMENT['analysis'],
            preprocessing_config={'features': {'degree': True, 'synapse_counts': True}},
            seed=seed,
            output_root=str(trial_out) if EXPERIMENT['export']['save_statistics'] else None,
            create_zip=EXPERIMENT['export']['create_zip'],
            extra={'metadata': EXPERIMENT['metadata']},
        )

        res = runner.run(config)
        results_per_rate[err_rate].append(res)

        if res.succeeded:
            print(f'  --> Success! Runtime: {res.runtime_seconds:.2f}s')
        else:
            print(f'  --> FAILED!  Errors: {res.errors}')

print('\nAll trials complete.')

In [ ]:
# Cell 8: Statistical Evaluation
evaluator = StatisticalEvaluator()
aggregated_stats_by_rate = {}

baseline_runs = [r for r in results_per_rate.get(0.00, []) if r.succeeded]
if not baseline_runs:
    raise RuntimeError('No successful baseline (0%) runs. Cannot evaluate.')

for err_rate, run_results in results_per_rate.items():
    successful = [r for r in run_results if r.succeeded]
    if successful:
        eval_result = evaluator.evaluate(baseline_runs, successful)
        aggregated_stats_by_rate[err_rate] = eval_result
        print(f'Evaluated {err_rate*100:.1f}%  -> {len(successful)} successful trials')
    else:
        print(f'Skipped   {err_rate*100:.1f}%  -> 0 successful trials')

print('\nStatistical evaluation complete.')

In [ ]:
# Cell 9: Export Presentation Layer
# Plots -> results/<DATASET>/missed_synapses/presentation/plots/
ExportManager().export_presentation(
    results_by_rate=aggregated_stats_by_rate,
    output_root=OUTPUT_ROOT,
    metadata=EXPERIMENT['metadata'],
)
print(f'Presentation exported to : {OUTPUT_ROOT / "presentation"}')
print(f'Plots saved to           : {OUTPUT_ROOT / "presentation" / "plots"}')

In [ ]:
# Cell 10: Summary Table
for err_rate, eval_result in sorted(aggregated_stats_by_rate.items()):
    print(f'\n{"="*60}')
    print(f'  Error Rate: {err_rate * 100:.1f}%')
    print(f'{"="*60}')
    for analysis_name, m_dict in eval_result.metrics.items():
        print(f'\n  Analysis: {analysis_name}')
        rows = []
        for m_name, ev in m_dict.items():
            rows.append({
                'Metric':          m_name,
                'Baseline Mean':   round(ev.baseline_mean, 4),
                'Perturbed Mean':  round(ev.mean, 4),
                'Std':             round(ev.std, 4),
                'CI Lower':        round(ev.ci_lower, 4),
                'CI Upper':        round(ev.ci_upper, 4),
                'Effect Size (d)': round(ev.effect_size, 4),
            })
        if rows:
            display(pd.DataFrame(rows))

In [ ]:
# Cell 11: Final Output Summary
print('\n' + '='*60)
print('  EXPERIMENT COMPLETE')
print('='*60)
print(f'  Dataset      : {DATASET_NAME}')
print(f'  Error Model  : {EXPERIMENT["error"]["name"]}')
print(f'  Rates Done   : {list(aggregated_stats_by_rate.keys())}')
print(f'\n  All outputs in : {OUTPUT_ROOT}/')
print(f'  |-- 0_percent/trial_001..N/')
print(f'  |-- 10_percent/trial_001..N/')
print(f'  |-- 20_percent/trial_001..N/')
print(f'  └── presentation/plots/')
if IS_KAGGLE:
    print(f'\n  Download: Kaggle Output panel > {OUTPUT_ROOT}')

In [ ]:
# Cell 12: Zip Results for Kaggle Download
import shutil
if IS_KAGGLE:
    zip_name = f"{EXPERIMENT['metadata']['experiment_name']}_results"
    zip_path = f"/kaggle/working/{zip_name}"
    print(f'Zipping {OUTPUT_ROOT} -> {zip_path}.zip ...')
    shutil.make_archive(zip_path, 'zip', OUTPUT_ROOT)
    print(f'Done! Download "{zip_name}.zip" from the Kaggle Output panel.')
else:
    print('Running locally — skipping zip (files already in results/ folder).')